# 08. RQ3 disability strand -- forward forecasting (Year 9-16)


## 0. Setup

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

FORECAST_START_YEAR = 8   # last observed year, forecasting begins at 9
FORECAST_HORIZON = 8      # Year 9 .. Year 16
YEAR_REF = 8               # keep the same time_trend scale used in training

DISABILITY_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed")
OUTPUT_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs")
FORECAST_DIR = OUTPUT_DIR / 'forecast_year9_16'
FORECAST_DIR.mkdir(parents=True, exist_ok=True)

summary = pd.read_csv(OUTPUT_DIR / 'summary_full.csv')

with open(OUTPUT_DIR / 'fitted_models.pkl', 'rb') as f:
    fitted_models = pickle.load(f)

print('tasks available in summary:', summary['task'].unique().tolist())
print('fitted models available:', list(fitted_models.keys()))

tasks available in summary: ['age_overall_level', 'disability_overall_level', 'disability_activity_level', 'disability_months12', 'disability_days10p60gr', 'age_months12', 'age_days10p60gr', 'age_activity_level']
fitted models available: ['age_overall__Ridge Regression', 'age_overall__Random Forest', 'age_overall__Gradient Boosting', 'dis_overall__Ridge Regression', 'dis_overall__Random Forest', 'dis_overall__Gradient Boosting', 'dis_level__Ridge Regression', 'dis_level__Random Forest', 'dis_level__Gradient Boosting', 'months12__Ridge Regression', 'months12__Random Forest', 'months12__Gradient Boosting', 'days__Ridge Regression', 'days__Random Forest', 'days__Gradient Boosting', 'age_months12__Ridge Regression', 'age_months12__Random Forest', 'age_months12__Gradient Boosting', 'age_days__Ridge Regression', 'age_days__Random Forest', 'age_days__Gradient Boosting', 'age_level__Ridge Regression', 'age_level__Random Forest', 'age_level__Gradient Boosting']


## 1. Shared forecasting functions


In [3]:
def alr_to_shares(values):
    values = np.clip(np.asarray(values, dtype=float), -30, 30)
    exponent = np.exp(values)
    denominator = 1 + exponent.sum(axis=1, keepdims=True)
    return np.column_stack([exponent / denominator, 1 / denominator])


def select_best_model(summary_df, task_name, metric_col, exclude_naive=True):
    """从summary_full.csv中按metric_col挑出该任务测试集表现最好的模型名称。
    Picks the model name with the lowest metric_col on the test split for the
    given task, matching the model naming used when fitted_models.pkl was saved."""
    subset = summary_df[summary_df['task'] == task_name]
    if exclude_naive:
        subset = subset[subset['model'] != 'Naive baseline']
    best_row = subset.loc[subset[metric_col].idxmin()]
    return best_row['model']

In [4]:
def forecast_composition_task(fitted_model, seed_frame, panel_keys, target_cols,
                               start_year=FORECAST_START_YEAR,
                               horizon=FORECAST_HORIZON, year_ref=YEAR_REF):
    lag_cols = [f'{c}_lag1' for c in target_cols]
    current = seed_frame[panel_keys + target_cols].copy()
    records = []

    for step in range(1, horizon + 1):
        forecast_year = start_year + step
        features = current[panel_keys].copy()
        for target_col, lag_col in zip(target_cols, lag_cols):
            features[lag_col] = current[target_col].to_numpy()
        features['time_trend'] = forecast_year / year_ref

        if fitted_model is None:
            predicted_shares = current[target_cols].to_numpy()
        else:
            predicted_alr = fitted_model.predict(features[lag_cols + ['time_trend'] + panel_keys])
            predicted_shares = alr_to_shares(predicted_alr)

        step_result = features[panel_keys].copy()
        step_result['year'] = forecast_year
        for index, target_col in enumerate(target_cols):
            step_result[target_col] = predicted_shares[:, index]
        records.append(step_result)
        current = step_result.copy()

    return pd.concat(records, ignore_index=True)

In [5]:
def forecast_single_rate_task(fitted_model, seed_frame, panel_keys, target_col,
                               start_year=FORECAST_START_YEAR,
                               horizon=FORECAST_HORIZON, year_ref=YEAR_REF):
    lag_col = f'{target_col}_lag1'
    current = seed_frame[panel_keys + [target_col]].copy()
    records = []

    for step in range(1, horizon + 1):
        forecast_year = start_year + step
        features = current[panel_keys].copy()
        features[lag_col] = current[target_col].to_numpy()
        features['time_trend'] = forecast_year / year_ref

        if fitted_model is None:
            predicted = current[target_col].to_numpy()
        else:
            predicted = np.clip(
                fitted_model.predict(features[[lag_col, 'time_trend'] + panel_keys]), 0, 1
            )

        step_result = features[panel_keys].copy()
        step_result['year'] = forecast_year
        step_result[target_col] = predicted
        records.append(step_result)
        current = step_result.copy()

    return pd.concat(records, ignore_index=True)

## 2. Forecast: disability overall activity level

Three-tier composition (inactive / fairly_active / active) at the
borough x disability_group level.

In [6]:
dis_overall_raw = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_overall_all_years.csv')
dis_overall_raw['LA_2023'] = dis_overall_raw['LA_2023'].astype('Int64').astype(str)

dis_overall_targets = ['inactive_rate', 'fairly_active_rate', 'active_rate']
dis_overall_panel_keys = ['LA_2023', 'disability_group']
dis_overall_seed = dis_overall_raw[dis_overall_raw['year'] == FORECAST_START_YEAR].dropna(subset=dis_overall_targets)

dis_overall_best_model_name = select_best_model(summary, 'disability_overall_level', 'total_variation')
dis_overall_best_model = fitted_models[f'dis_overall__{dis_overall_best_model_name}']
print('best model, disability overall level:', dis_overall_best_model_name)

dis_overall_forecast = forecast_composition_task(
    dis_overall_best_model, dis_overall_seed,
    panel_keys=dis_overall_panel_keys, target_cols=dis_overall_targets,
)
dis_overall_forecast_naive = forecast_composition_task(
    None, dis_overall_seed,
    panel_keys=dis_overall_panel_keys, target_cols=dis_overall_targets,
)
dis_overall_forecast.head()

best model, disability overall level: Gradient Boosting


,LA_2023,disability_group,year,inactive_rate,fairly_active_rate,active_rate
0,8,limiting_disability,9,0.400125,0.104755,0.495120
1,8,non_limiting_disability,9,0.259831,0.116193,0.623976
2,8,no_disability,9,0.281972,0.098211,0.619817
3,8,disty1,9,0.422144,0.097923,0.479933
4,8,disty2,9,0.401311,0.104036,0.494653


## 3. Forecast: disability activity-specific level

Same three-tier composition, but at the borough x disability_group x activity
granularity.

In [7]:
dis_level_raw = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_all_years.csv')
dis_level_raw['LA_2023'] = dis_level_raw['LA_2023'].astype('Int64').astype(str)

dis_level_targets = ['inactive_rate', 'fairly_active_rate', 'active_rate']
dis_level_panel_keys = ['LA_2023', 'disability_group', 'activity']
dis_level_seed = dis_level_raw[dis_level_raw['year'] == FORECAST_START_YEAR].dropna(subset=dis_level_targets)

dis_level_best_model_name = select_best_model(summary, 'disability_activity_level', 'total_variation')
dis_level_best_model = fitted_models[f'dis_level__{dis_level_best_model_name}']
print('best model, disability activity level:', dis_level_best_model_name)

dis_level_forecast = forecast_composition_task(
    dis_level_best_model, dis_level_seed,
    panel_keys=dis_level_panel_keys, target_cols=dis_level_targets,
)
dis_level_forecast.head()

best model, disability activity level: Random Forest


,LA_2023,disability_group,activity,year,inactive_rate,fairly_active_rate,active_rate
0,8,limiting_disability,ABSEILING_H03,9,0.999997,0.000001,0.000001
1,8,non_limiting_disability,ABSEILING_H03,9,0.999997,0.000001,0.000001
2,8,no_disability,ABSEILING_H03,9,0.999996,0.000002,0.000002
3,8,disty1,ABSEILING_H03,9,0.999997,0.000001,0.000001
4,8,disty2,ABSEILING_H03,9,0.999997,0.000001,0.000001


## 4. Forecast: MONTHS_12 and DAYS10P60GR participation rate

The two binary participation measures at the borough x disability_group x
activity level, modelled and forecast independently.

In [8]:
dis_dm_raw = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_days_months_all_years.csv')
dis_dm_raw['LA_2023'] = dis_dm_raw['LA_2023'].astype('Int64').astype(str)
dis_dm_panel_keys = ['LA_2023', 'disability_group', 'activity']

months12_seed = dis_dm_raw[dis_dm_raw['year'] == FORECAST_START_YEAR].dropna(subset=['participation_MONTHS_12'])
months12_best_model_name = select_best_model(summary, 'disability_months12', 'participation_MONTHS_12_mae')
months12_best_model = fitted_models[f'months12__{months12_best_model_name}']
print('best model, MONTHS_12:', months12_best_model_name)

months12_forecast = forecast_single_rate_task(
    months12_best_model, months12_seed,
    panel_keys=dis_dm_panel_keys, target_col='participation_MONTHS_12',
)

days_seed = dis_dm_raw[dis_dm_raw['year'] == FORECAST_START_YEAR].dropna(subset=['participation_DAYS10P60GR'])
days_best_model_name = select_best_model(summary, 'disability_days10p60gr', 'participation_DAYS10P60GR_mae')
days_best_model = fitted_models[f'days__{days_best_model_name}']
print('best model, DAYS10P60GR:', days_best_model_name)

days_forecast = forecast_single_rate_task(
    days_best_model, days_seed,
    panel_keys=dis_dm_panel_keys, target_col='participation_DAYS10P60GR',
)

months12_forecast.head()

best model, MONTHS_12: Ridge Regression
best model, DAYS10P60GR: Gradient Boosting


,LA_2023,disability_group,activity,year,participation_MONTHS_12
0,8,limiting_disability,ABSEILING_H03,9,0.000000
1,8,non_limiting_disability,ABSEILING_H03,9,0.008782
2,8,no_disability,ABSEILING_H03,9,0.014541
3,8,disty1,ABSEILING_H03,9,0.000000
4,8,disty2,ABSEILING_H03,9,0.000000


## 5. Most preferred activity, by forecast year

Same logic as most_preferred_activity in notebook 06, applied within each
forecast year: for every borough x disability_group, take the activity with
the highest predicted MONTHS_12 participation rate.

In [9]:
def top_activity_by_year(forecast_frame, id_cols, activity_col, value_col, top_n=1):
    ranked = forecast_frame.sort_values(
        id_cols + [value_col], ascending=[True] * len(id_cols) + [False]
    )
    return ranked.groupby(id_cols, as_index=False).head(top_n)[
        id_cols + [activity_col, value_col]
    ]


top_activity_forecast = top_activity_by_year(
    months12_forecast,
    id_cols=['year', 'LA_2023', 'disability_group'],
    activity_col='activity',
    value_col='participation_MONTHS_12',
)
top_activity_forecast.head(10)

,year,LA_2023,disability_group,activity,participation_MONTHS_12
659,9,107,disty1,ACTTRAV_C03,0.585201
668,9,107,disty10,ACTTRAV_C03,0.591960
669,9,107,disty11,ACTTRAV_C03,0.540232
670,9,107,disty12,ACTTRAV_C03,0.600062
671,9,107,disty13,ACTTRAV_C03,0.626418
660,9,107,disty2,ACTTRAV_C03,0.588848
661,9,107,disty3,ACTTRAV_C03,0.559671
662,9,107,disty4,ACTTRAV_C03,0.550952
663,9,107,disty5,ACTTRAV_C03,0.601964
664,9,107,disty6,ACTTRAV_C03,0.543635


## 6. Sanity check: model forecast vs naive persistence

In [10]:
comparison = dis_overall_forecast.merge(
    dis_overall_forecast_naive,
    on=['LA_2023', 'disability_group', 'year'],
    suffixes=('_model', '_naive'),
)
comparison['active_rate_gap'] = (comparison['active_rate_model'] - comparison['active_rate_naive']).abs()

gap_by_year = comparison.groupby('year')['active_rate_gap'].mean()
print('mean |model - naive| gap in active_rate, by forecast year:')
print(gap_by_year.round(4))

mean |model - naive| gap in active_rate, by forecast year:
year
9     0.1286
10    0.1494
11    0.1544
12    0.1547
13    0.1551
14    0.1550
15    0.1552
16    0.1550
Name: active_rate_gap, dtype: float64


## 7. Save forecast outputs

In [11]:
dis_overall_forecast.to_csv(FORECAST_DIR / 'forecast_disability_overall_level.csv', index=False)
dis_overall_forecast_naive.to_csv(FORECAST_DIR / 'forecast_disability_overall_level_naive.csv', index=False)
dis_level_forecast.to_csv(FORECAST_DIR / 'forecast_disability_activity_level.csv', index=False)
months12_forecast.to_csv(FORECAST_DIR / 'forecast_disability_months12.csv', index=False)
days_forecast.to_csv(FORECAST_DIR / 'forecast_disability_days10p60gr.csv', index=False)
top_activity_forecast.to_csv(FORECAST_DIR / 'forecast_top_activity_by_year.csv', index=False)

print('Saved to', FORECAST_DIR)
print('- forecast_disability_overall_level.csv')
print('- forecast_disability_overall_level_naive.csv')
print('- forecast_disability_activity_level.csv')
print('- forecast_disability_months12.csv')
print('- forecast_disability_days10p60gr.csv')
print('- forecast_top_activity_by_year.csv')

Saved to C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs\forecast_year9_16
- forecast_disability_overall_level.csv
- forecast_disability_overall_level_naive.csv
- forecast_disability_activity_level.csv
- forecast_disability_months12.csv
- forecast_disability_days10p60gr.csv
- forecast_top_activity_by_year.csv
